# Demo: Analisi emotiva di un singolo commento con ELIta

Questo notebook mostra **passo per passo** come il metodo finale assegna un'emozione a un commento del corpus `r/Italia — notizie`.

**Metodo finale**: lessico ELIta ibrido (α=0.5) + ALL_STOPWORDS + soglia di distintività ≥ 0.06

Per ogni commento selezionato:
1. Si mostrano i token lemmatizzati e quali vengono usati
2. Si mostra il contributo emotivo di ogni parola trovata in ELIta
3. Si calcola il vettore emotivo aggregato e l'emozione dominante

## Setup e caricamento dati

In [1]:
import pandas as pd
import numpy as np
import emoji
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
from IPython.display import display

CORPUS_CSV   = Path('corpus_Italia_notizie.csv')
TOKENS_CSV   = Path('tokens_Italia_notizie.csv')
ELITA_CSV    = Path('../Fase1/ELIta_INTENSITY_Matrix.csv')
ALPHA_02_CSV = Path('../Fase2/output_csv/elita_recalculated_0_2.csv')
ALPHA_05_CSV = Path('../Fase2/output_csv/elita_recalculated_0_5.csv')
ALPHA_08_CSV = Path('../Fase2/output_csv/elita_recalculated_0_8.csv')

BASIC_EMOTIONS = ['gioia','tristezza','rabbia','paura','disgusto','fiducia','sorpresa','aspettativa']
EMOTION_COLORS = {
    'gioia':'#FDD835', 'tristezza':'#1E88E5', 'rabbia':'#E53935', 'paura':'#43A047',
    'disgusto':'#8E24AA', 'fiducia':'#81C784', 'sorpresa':'#039BE5',
    'aspettativa':'#FB8C00', 'neutrale':'#9E9E9E',
}

# Stopwords (stesse di Confronto_notizie.ipynb)
EMOTIONAL_STOPWORDS = {
    'avere','essere','fare','stare','dare','andare','venire',
    'potere','volere','dovere','sapere','vedere','sentire',
    'trovare','pensare','dire','parlare','guardare','tenere',
    'portare','prendere','mettere','lasciare','passare','uscire',
    'entrare','tornare','rimanere','iniziare','finire','continuare',
    'cominciare','provare','riuscire','sembrare','diventare',
    'cosa','modo','parte','punto','volta','anno','tempo','caso',
    'fatto','posto','tipo','gente','persona','vita','mondo',
    'uomo','donna','bambino','figlio','figlia','padre','madre',
    'altro','solo','grande','piccolo','nuovo','vecchio','primo',
    'ultimo','stesso','proprio','bello','buono','lungo','alto',
    'più','bene','male','molto','poco','tanto','tutto','niente',
}
TOPIC_STOPWORDS = {
    'notizia','notizie','giornale','giornali','giornalista','giornalismo',
    'informazione','informazioni','articolo','articoli','media','fonte',
    'fonti','testata','redazione','titolo','telegiornale',
}
ALL_STOPWORDS = EMOTIONAL_STOPWORDS | TOPIC_STOPWORDS
DIST_THRESHOLD = 0.06
POS_FILTER     = {'ADJ', 'NOUN', 'VERB'}

print('Configurazione caricata.')

Configurazione caricata.


In [2]:
df_corpus = pd.read_csv(CORPUS_CSV)
df_tokens = pd.read_csv(TOKENS_CSV)
df_tokens['lemma'] = df_tokens['lemma'].astype(str).str.lower().str.strip()
df_tokens['pos']   = df_tokens['pos'].astype(str).str.upper().str.strip()

def is_not_emoji(text):
    return emoji.emoji_count(str(text)) == 0

# Metodo finale: solo α=0.5
df_elita_final = pd.read_csv(ALPHA_05_CSV, index_col=0)
df_elita_final.index = df_elita_final.index.astype(str).str.lower().str.strip()
df_elita_final = df_elita_final[BASIC_EMOTIONS].fillna(0)

# Pre-calcolo distintività sul lessico originale (stessa soglia di Confronto_notizie.ipynb)
df_elita_orig = pd.read_csv(ELITA_CSV, index_col=0)
df_elita_orig = df_elita_orig[df_elita_orig.index.map(is_not_emoji)]
df_elita_orig.index = df_elita_orig.index.astype(str).str.lower().str.strip()
df_elita_orig = df_elita_orig[BASIC_EMOTIONS].fillna(0)

def calc_dist(df_e):
    def _row(row):
        sv = sorted(row.values, reverse=True)
        max1, max2 = sv[0], sv[1]
        mn = np.mean(row.values)
        if max1 == 0:
            return 0.0
        return ((max1 - max2) / (max1 + 1e-9)) * (max1 - mn)
    return df_e[BASIC_EMOTIONS].apply(_row, axis=1)

dist_orig = calc_dist(df_elita_orig)
ELITA_IDX_FILTERED = set(dist_orig[dist_orig >= DIST_THRESHOLD].index)

print(f'Corpus: {len(df_corpus)} commenti | Token: {len(df_tokens)}')
print(f'Parole ELIta (α=0.5) con dist ≥ {DIST_THRESHOLD}: {len(ELITA_IDX_FILTERED)}')

Corpus: 700 commenti | Token: 64812
Parole ELIta (α=0.5) con dist ≥ 0.06: 3107


## Selezione del commento

Modifica `COMMENT_INDEX` (0–699) per scegliere un commento diverso, oppure imposta `COMMENT_ID` con un ID specifico.

In [3]:
COMMENT_INDEX = 492      # <--- cambia qui (0-699)
COMMENT_ID    = None   # oppure specifica un ID, es. 'kfr4gvl'

if COMMENT_ID:
    row = df_corpus[df_corpus['comment_id'] == COMMENT_ID].iloc[0]
else:
    row = df_corpus.iloc[COMMENT_INDEX]

CID  = row['comment_id']
TEXT = row['comment_text']

print(f'ID commento : {CID}')
print(f'Autore      : {row["comment_author"]}')
print(f'Score Reddit: {row["comment_score"]}')
print()
print('Testo:')
print('-' * 70)
print(TEXT)
print('-' * 70)

ID commento : iwyec0t
Autore      : TestaOnFire
Score Reddit: 8

Testo:
----------------------------------------------------------------------
Vi rendente conto che ha più volte incitato alla violenza, diffuso false notizie e pure pubblicato informazioni secretate su Tweeter vero?
----------------------------------------------------------------------


## Token e lemmi del commento

In [4]:
df_tok = df_tokens[df_tokens['comment_id'] == CID].copy()

elita_idx_all = set(df_elita_final.index)

df_tok['in_ELIta']  = df_tok['lemma'].isin(elita_idx_all)
df_tok['pos_ok']    = df_tok['pos'].isin(POS_FILTER)
df_tok['stopword']  = df_tok['lemma'].isin(ALL_STOPWORDS)
df_tok['dist_ok']   = df_tok['lemma'].isin(ELITA_IDX_FILTERED)
df_tok['usato']     = df_tok['in_ELIta'] & df_tok['pos_ok'] & ~df_tok['stopword'] & df_tok['dist_ok']

print(f'Token totali nel commento      : {len(df_tok)}')
print(f'Con POS valida (ADJ/NOUN/VERB) : {df_tok["pos_ok"].sum()}')
print(f'Trovati in ELIta               : {df_tok["in_ELIta"].sum()}')
print(f'Stopwords rimosse              : {(df_tok["in_ELIta"] & df_tok["stopword"]).sum()}')
print(f'Sotto soglia distintività      : {(df_tok["in_ELIta"] & df_tok["pos_ok"] & ~df_tok["stopword"] & ~df_tok["dist_ok"]).sum()}')
print(f'Token usati per l\'analisi      : {df_tok["usato"].sum()}')
print()

display(df_tok[['token','lemma','pos','in_ELIta','stopword','dist_ok','usato']].reset_index(drop=True))

Token totali nel commento      : 21
Con POS valida (ADJ/NOUN/VERB) : 12
Trovati in ELIta               : 11
Stopwords rimosse              : 5
Sotto soglia distintività      : 5
Token usati per l'analisi      : 1



,token,lemma,pos,in_ELIta,stopword,dist_ok,usato
0,Vi,vi,PRON,False,False,False,False
1,rendente,rendente,ADJ,False,False,False,False
2,conto,conto,NOUN,True,False,False,False
3,che,che,PRON,False,False,False,False
4,ha,avere,AUX,True,True,False,False
5,più,più,ADV,True,True,True,False
6,volte,volta,NOUN,True,True,False,False
7,incitato,incitare,VERB,False,False,False,False
8,alla,a il,ADP,False,False,False,False
9,violenza,violenza,NOUN,True,False,False,False


## Contributo emotivo per parola

Per ogni lemma usato nell'analisi, mostriamo il vettore emotivo da ELIta (versione originale).

In [5]:
lemmi_usati = df_tok[df_tok['usato']]['lemma'].tolist()

if not lemmi_usati:
    print('Nessun lemma utile trovato in questo commento.')
else:
    contrib_rows = []
    for lemma in lemmi_usati:
        scores = df_elita_final.loc[lemma, BASIC_EMOTIONS].to_dict()
        dom    = max(scores, key=scores.get)
        scores['lemma']   = lemma
        scores['dom_emo'] = dom
        contrib_rows.append(scores)

    df_contrib = pd.DataFrame(contrib_rows)
    cols_show  = ['lemma'] + BASIC_EMOTIONS + ['dom_emo']

    totals  = df_contrib[BASIC_EMOTIONS].sum()
    dom_tot = totals.idxmax()

    print('Contributi emotivi per lemma (ELIta α=0.5):')
    display(
        df_contrib[cols_show]
        .style
        .background_gradient(subset=BASIC_EMOTIONS, cmap='YlOrRd', axis=None)
        .format({e: '{:.2f}' for e in BASIC_EMOTIONS})
    )
    print()
    print('TOTALE per emozione:')
    print(totals.round(3).to_string())
    print(f'\n=> Emozione dominante: {dom_tot.upper()}')

Contributi emotivi per lemma (ELIta α=0.5):


,lemma,gioia,tristezza,rabbia,paura,disgusto,fiducia,sorpresa,aspettativa,dom_emo
0,diffuso,0.20,0.30,0.25,0.42,0.33,0.22,0.31,0.57,aspettativa



TOTALE per emozione:
gioia          0.197
tristezza      0.297
rabbia         0.253
paura          0.422
disgusto       0.332
fiducia        0.218
sorpresa       0.306
aspettativa    0.571

=> Emozione dominante: ASPETTATIVA


## Profilo emotivo aggregato — radar chart

In [6]:
if not lemmi_usati:
    print('Nessun lemma utile trovato.')
else:
    emos_loop = BASIC_EMOTIONS + [BASIC_EMOTIONS[0]]
    vals = [totals[e] for e in emos_loop]

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=vals, theta=emos_loop,
        fill='toself', opacity=0.6,
        name='ELIta α=0.5',
        line_color='#FB8C00'
    ))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True)),
        title=f'Profilo emotivo — commento {CID} (metodo finale)',
        height=480
    )
    fig.show()

## Barchart: score per emozione

In [7]:
if not lemmi_usati:
    print('Nessun lemma utile trovato.')
else:
    bar_colors = [EMOTION_COLORS[e] for e in BASIC_EMOTIONS]
    fig = go.Figure(go.Bar(
        x=BASIC_EMOTIONS,
        y=[totals[e] for e in BASIC_EMOTIONS],
        marker_color=bar_colors,
        text=[f'{totals[e]:.2f}' for e in BASIC_EMOTIONS],
        textposition='outside'
    ))
    fig.update_layout(
        title=f'Score emotivi aggregati — commento {CID} (ELIta α=0.5)',
        xaxis_title='Emozione',
        yaxis_title='Score aggregato',
        height=430
    )
    fig.show()

    print(f'\nLemmi usati ({len(lemmi_usati)}): {lemmi_usati}')
    print(f'Emozione dominante: {dom_tot.upper()} (score: {totals[dom_tot]:.3f})')


Lemmi usati (1): ['diffuso']
Emozione dominante: ASPETTATIVA (score: 0.571)
